# Display difference in admissions for the Taunton+Worcester case

Where do patients go when the units change?

## Code setup

In [16]:
import os
import numpy as np
import pandas as pd

In [17]:
dir_data = 'data'
dir_output = 'output'
# Reference:
file_hospitals = 'stroke_hospitals_2025.csv'
# Output:
file_admissions_by_unit = 'results_unit_admissions_base_taunworc.csv'
file_admissions_by_transfer_unit = 'results_mt_unit_admissions_base_taunworc.csv'

In [48]:
scenarios = ['base', 'taunworc']

mt_rate = 0.1
transfer_rate = 0.6

col_admissions = 'Admissions_2025_Est'
cols_postcode = sum([[f'nearest_ivt_unit_{s}', f'transfer_unit_{s}']
                     for s in scenarios], [])

## Load data

Scenario results by unit:

In [21]:
p = os.path.join(dir_output, file_admissions_by_unit)
df_admissions_unit = pd.read_csv(p)

In [22]:
display(df_admissions_unit[df_admissions_unit['change']])

,nearest_ivt_unit_base,transfer_unit_base,nearest_ivt_unit_taunworc,transfer_unit_taunworc,Admissions_2025_Est,transfer_required_base,transfer_required_taunworc,change,mt_base,mt_transferloss_base,mt_taunworc,mt_transferloss_taunworc
26,DT12JY,SO166YD,DT12JY,TA15DA,697.8,True,True,True,69.8,41.9,69.8,41.9
27,DT12JY,SO166YD,TA15DA,TA15DA,6.3,True,False,True,0.6,0.4,0.6,0.6
29,DY12HQ,B152TH,WR51DD,WR51DD,4.9,True,False,True,0.5,0.3,0.5,0.5
31,EX25DW,PL68DH,EX25DW,TA15DA,628.1,True,True,True,62.8,37.7,62.8,37.7
32,EX314JB,PL68DH,EX314JB,TA15DA,467.4,True,True,True,46.7,28.0,46.7,28.0
34,GL537AN,BS105NB,GL537AN,WR51DD,797.5,True,True,True,79.8,47.9,79.8,47.9
38,HR12ER,B152TH,HR12ER,WR51DD,516.8,True,True,True,51.7,31.0,51.7,31.0
90,TA15DA,BS105NB,TA15DA,TA15DA,934.9,True,False,True,93.5,56.1,93.5,93.5
98,WR51DD,B152TH,WR51DD,WR51DD,700.4,True,False,True,70.0,42.0,70.0,70.0


Scenario results by transfer unit:

In [41]:
p = os.path.join(dir_output, file_admissions_by_transfer_unit)
df_admissions_mt_unit = pd.read_csv(p, index_col=0)

In [42]:
display(df_admissions_mt_unit[df_admissions_mt_unit['mt_change'] > 0.0])

,mt_base,mt_taunworc,mt_transferloss_base,mt_transferloss_taunworc,mt_change,mt_transferloss_change
TA15DA,0.0,273.4,0.0,201.7,273.4,201.7
WR51DD,0.0,202.0,0.0,149.4,202.0,149.4


### Hospital reference

In [25]:
# Load hospital coordinates from CSV and create a GeoDataFrame
p = os.path.join(dir_data, file_hospitals)
df_hospitals = pd.read_csv(p, index_col='Postcode')

# Only keep hospitals in England:
df_hospitals = df_hospitals.loc[df_hospitals["Country"] == "England"]

In [26]:
display(df_hospitals.head(3))

,SSNAP name,Country,SSNAP Region,Easting,Northing,long,lat,Hospital_City,LSOA11CD,Use_IVT,Use_MT
Postcode,,,,,,,,,,,
OX39DU,John Radcliffe Hospital,England,"Buckinghamshire, Oxford and Berkshire",453940,207531,-1.219806,51.763871,Oxford,E01028534,1,1
RG15AN,Royal Berkshire Hospital,England,"Buckinghamshire, Oxford and Berkshire",472411,172962,-0.959330,51.451022,Reading,E01016414,1,0
HP112TT,Wycombe General Hospital,England,"Buckinghamshire, Oxford and Berkshire",486386,192694,-0.753428,51.626442,High Wycombe,E01017839,1,0


In [27]:
mt_units = df_hospitals[df_hospitals['Use_MT'] == 1].index.tolist()
postcodes_taunworc = ['TA15DA', 'WR51DD']
mt_units_taunworc = mt_units + postcodes_taunworc

## Units affected

Gather the names of the units affected by the change.

IVT units affected:

In [28]:
units_affected = df_admissions_unit.loc[
    df_admissions_unit['change'], 'nearest_ivt_unit_base'].unique()

# Quick check that there are no extra units in other scenario:
units_affected2 = df_admissions_unit.loc[
    df_admissions_unit['change'], 'nearest_ivt_unit_taunworc'].unique()

print((set(units_affected) - set(units_affected2)))

{'DY12HQ'}


There is one unit that is different for 4.9 patients only, so ignore it.

In [29]:
units_affected = units_affected2

Anyone who goes to an affected unit as their first unit in the base case:

In [30]:
display(df_hospitals.loc[units_affected, ['SSNAP name', 'Hospital_City']])

,SSNAP name,Hospital_City
Postcode,,
DT12JY,Dorset County Hospital,Dorchester
TA15DA,Musgrove Park Hospital,Taunton
WR51DD,Worcestershire Royal Hospital,Worcester
EX25DW,Royal Devon and Exeter Hospital,Exeter
EX314JB,North Devon District Hospital,Barnstaple
GL537AN,Cheltenham & Gloucester Hospitals,Cheltenham
HR12ER,Hereford County Hospital,Hereford


MT units affected:

In [31]:
mt_units_affected = sorted(list(
    set(df_admissions_unit.loc[
        df_admissions_unit['change'], 'transfer_unit_base'].unique()) |
    set(df_admissions_unit.loc[
        df_admissions_unit['change'], 'transfer_unit_taunworc'].unique())
))

print(mt_units_affected)

['B152TH', 'BS105NB', 'PL68DH', 'SO166YD', 'TA15DA', 'WR51DD']


In [32]:
display(df_hospitals.loc[mt_units_affected, ['SSNAP name', 'Hospital_City']])

,SSNAP name,Hospital_City
Postcode,,
B152TH,Queen Elizabeth Hospital Edgbaston,Birmingham
BS105NB,Bristol and Weston SU,North Bristol
PL68DH,Derriford Hospital,Plymouth
SO166YD,Southampton General Hospital,Southampton
TA15DA,Musgrove Park Hospital,Taunton
WR51DD,Worcestershire Royal Hospital,Worcester


## Total patients affected

In [33]:
cols = ['Admissions_2025_Est', 'mt_base', 'mt_taunworc',
        'mt_transferloss_base', 'mt_transferloss_taunworc']
display(df_admissions_unit.loc[df_admissions_unit['change'], cols].sum())

Admissions_2025_Est         4754.1
mt_base                      475.4
mt_taunworc                  475.4
mt_transferloss_base         285.3
mt_transferloss_taunworc     351.1
dtype: float64

## Proportion going directly to MT-capable unit

In [34]:
def calculate_props(df):
    r = pd.Series()
    r['all'] = df[col_admissions].sum()
    for s in scenarios:
        r[f'direct_{s}'] = df.loc[~df[f'transfer_required_{s}'], col_admissions].sum()
        r[f'transfer_{s}'] = df.loc[df[f'transfer_required_{s}'], col_admissions].sum()
        r[f'prop_direct_{s}'] = (r[f'direct_{s}'] / r['all']).round(3)
        r[f'prop_transfer_{s}'] = (r[f'transfer_{s}'] / r['all']).round(3)
    return r

Whole of England:

In [35]:
calculate_props(df_admissions_unit)

all                       90659.000
direct_base               29991.500
transfer_base             60667.500
prop_direct_base              0.331
prop_transfer_base            0.669
direct_taunworc           31638.000
transfer_taunworc         59021.000
prop_direct_taunworc          0.349
prop_transfer_taunworc        0.651
dtype: float64

Only units affected by the change in MT units:

In [36]:
m = df_admissions_unit['nearest_ivt_unit_base'].isin(units_affected)
calculate_props(df_admissions_unit.loc[m])

all                       4749.200
direct_base                  0.000
transfer_base             4749.200
prop_direct_base             0.000
prop_transfer_base           1.000
direct_taunworc           1641.600
transfer_taunworc         3107.600
prop_direct_taunworc         0.346
prop_transfer_taunworc       0.654
dtype: float64

## Display change in admissions between scenarios

Rename dataframe labels for better display:

In [43]:
cols_transferloss = [
    c for c in df_admissions_mt_unit.columns if 'transferloss' in c]
cols_plain = [
    c for c in df_admissions_mt_unit.columns if 'transferloss' not in c]

rename_dict = {
    'mt_base': 'Current MT',
    'mt_taunworc': 'Planned MT',
    'mt_change': 'Change',
    'mt_transferloss_base': 'Current MT',
    'mt_transferloss_taunworc': 'Planned MT',
    'mt_transferloss_change': 'Change'
}

display_df_mt = (
    df_admissions_mt_unit[cols_plain]
        .rename(columns=rename_dict).copy())
display_df_mt_transferloss = (
    df_admissions_mt_unit[cols_transferloss]
        .rename(columns=rename_dict).copy()
)

display_df_mt.index = (
    display_df_mt.index.map(df_hospitals['Hospital_City']))
display_df_mt_transferloss.index = (
    display_df_mt_transferloss.index.map(df_hospitals['Hospital_City']))

mt_cities_affected = (
    df_hospitals.loc[mt_units_affected, 'Hospital_City'].tolist())

Display numbers of thrombectomy:

In [44]:
with pd.option_context('display.precision', 0):
    print('10% of all stroke receive MT, with no loss by transfer:')
    display(display_df_mt.loc[mt_cities_affected].round(0))
    print('')
    print('10% of all stroke receive MT, with 60% of patients transferred:')
    display(display_df_mt_transferloss.loc[mt_cities_affected].round(0))

10% of all stroke receive MT, with no loss by transfer:


,Current MT,Planned MT,Change
Birmingham,709,587,-122
North Bristol,393,219,-173
Plymouth,341,232,-110
Southampton,559,488,-70
Taunton,0,273,273
Worcester,0,202,202



10% of all stroke receive MT, with 60% of patients transferred:


,Current MT,Planned MT,Change
Birmingham,460,386,-73
North Bristol,289,185,-104
Plymouth,238,172,-66
Southampton,368,326,-42
Taunton,0,202,202
Worcester,0,149,149


Check where the admissions come from:

In [45]:
sum([[f'nearest_ivt_unit_{s}', f'transfer_unit_{s}'] for s in scenarios], [])

['nearest_ivt_unit_base',
 'transfer_unit_base',
 'nearest_ivt_unit_taunworc',
 'transfer_unit_taunworc']

In [49]:
m = (df_admissions_unit['transfer_unit_taunworc'].isin(postcodes_taunworc) &
     ~df_admissions_unit['nearest_ivt_unit_base'].isin(postcodes_taunworc))

# Display with hospital cities instead of postcodes:
display_mt_changed = df_admissions_unit.copy()
for c in cols_postcode:
    display_mt_changed[c] = display_mt_changed[c].map(
        df_hospitals['Hospital_City'])
display(display_mt_changed.loc[m].sort_values('transfer_unit_taunworc'))

,nearest_ivt_unit_base,transfer_unit_base,nearest_ivt_unit_taunworc,transfer_unit_taunworc,Admissions_2025_Est,transfer_required_base,transfer_required_taunworc,change,mt_base,mt_transferloss_base,mt_taunworc,mt_transferloss_taunworc
26,Dorchester,Southampton,Dorchester,Taunton,697.8,True,True,True,69.8,41.9,69.8,41.9
27,Dorchester,Southampton,Taunton,Taunton,6.3,True,False,True,0.6,0.4,0.6,0.6
31,Exeter,Plymouth,Exeter,Taunton,628.1,True,True,True,62.8,37.7,62.8,37.7
32,Barnstaple,Plymouth,Barnstaple,Taunton,467.4,True,True,True,46.7,28.0,46.7,28.0
29,Dudley,Birmingham,Worcester,Worcester,4.9,True,False,True,0.5,0.3,0.5,0.5
34,Cheltenham,North Bristol,Cheltenham,Worcester,797.5,True,True,True,79.8,47.9,79.8,47.9
38,Hereford,Birmingham,Hereford,Worcester,516.8,True,True,True,51.7,31.0,51.7,31.0


From the `mt_base` and `mt_transferloss_base` columns:
+ About 70 patients have nearest unit Dorcester and switch to transfer unit Taunton (from base Southampton). In the lossy case, about 42 are transferred for MT.
+ About 80 patients have nearest unit Cheltenham and switch to transfer unit Worcester (from base North Bristol). In the lossy case, about 48 are transferred for MT.